In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/30 18:47:13 WARN Utils: Your hostname, xicrth resolves to a loopback address: 127.0.1.1; using 10.210.69.94 instead (on interface wlp62s0)
26/09/30 18:47:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 18:47:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [4]:
df_transaksi_etl = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_pelanggan_etl = spark.read.json("data_pelanggan.json")
print("Siap untuk latihan.")

Siap untuk latihan.


In [5]:
df_parquet_latihan = spark.read.parquet("data_transaksi_besar_parquet")
print("Skema dari Parquet:")
df_parquet_latihan.printSchema()
print("Skema dari CSV (inferSchema=True):")
df_transaksi_etl.printSchema()

Skema dari Parquet:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)

Skema dari CSV (inferSchema=True):
root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)



In [6]:
df_gabung_latihan = df_transaksi_etl.join(df_pelanggan_etl, on="customer_id", how="left")
df_gabung_latihan.groupBy("membership") \
    .agg(count("order_id").alias("jumlah_transaksi")) \
    .orderBy(col("jumlah_transaksi").desc()) \
    .show()

+----------+----------------+
|membership|jumlah_transaksi|
+----------+----------------+
|    Silver|            8994|
|      Gold|            6990|
|  Platinum|            4016|
+----------+----------------+



In [8]:
df_gabung_latihan.write.mode("overwrite") \
    .partitionBy("membership") \
    .parquet("data_transaksi_partisi_membership")

!ls data_transaksi_partisi_membership

'membership=Gold'  'membership=Platinum'  'membership=Silver'   _SUCCESS
